<a href="https://colab.research.google.com/github/mindra-bit/DSDPM/blob/main/DSDPM_COLAB_MULTICORE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DSDPM v3.4 — Colab Multicore R=500

Notebook ini menjalankan **5 batch × 100 replikasi** dengan beberapa proses R secara paralel.

**Sebelum mulai**, upload ke Google Drive:

1. ZIP **paket DSDPM v3.4 lengkap/original** (harus berisi `MASTER_SOURCE.R`, folder `R`, `RUN_RUNTIME_SELFTEST.R`, dan folder `tests`).
2. ZIP **`DSDPM_v3_4_COLAB_MULTICORE.zip`** dari ChatGPT.

Disarankan memilih **Runtime → Change runtime type → CPU / Hardware accelerator: None**. GPU tidak diperlukan untuk QMLE/GMM/INLA pada kode ini.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 1. Atur lokasi file

Ubah `BASE_ZIP` bila nama ZIP paket DSDPM asli Anda berbeda. Folder persistent akan menyimpan environment lock, checkpoint batch, dan hasil final.


In [3]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

for p in drive.rglob("*DSDPM*"):
    print(p)

/content/drive/MyDrive/@DSDPM
/content/drive/MyDrive/DSDPM_COLAB
/content/drive/MyDrive/Colab Notebooks/DSDPM_COLAB_MULTICORE.ipynb
/content/drive/MyDrive/DSDPM_COLAB/input/DSDPM_Reviewer_Revision_v3_4
/content/drive/MyDrive/DSDPM_COLAB/input/DSDPM_v3_4_COLAB_MULTICORE.zip
/content/drive/MyDrive/DSDPM_COLAB/input/DSDPM_COLAB_MULTICORE.ipynb
/content/drive/MyDrive/DSDPM_COLAB/input/DSDPM_COLAB_MULTICORE_FINAL
/content/drive/MyDrive/DSDPM_COLAB/input/DSDPM_Reviewer_Revision_v3_4/DSDPM_ALL_IN_ONE_v3_4.R
/content/drive/MyDrive/DSDPM_COLAB/input/DSDPM_COLAB_MULTICORE_FINAL/DSDPM_COLAB_MULTICORE.ipynb
/content/drive/MyDrive/DSDPM_COLAB/input/DSDPM_Reviewer_Revision_v3_4/source_basis/Paper DSDPM (Re-Submitted)(Main)(3).docx


In [4]:
from pathlib import Path
import os, shutil, zipfile, subprocess, tarfile, json, glob

DRIVE_ROOT = Path("/content/drive/MyDrive/DSDPM_COLAB")
INPUT_DIR = DRIVE_ROOT / "input"
PERSIST_ROOT = DRIVE_ROOT / "persistent"

BASE_ZIP = INPUT_DIR / "DSDPM_Reviewer_Revision_v3_4.zip"
ADDON_ZIP = INPUT_DIR / "DSDPM_v3_4_COLAB_MULTICORE.zip"

WORK_ROOT = Path("/content/dsdpm_work")
RLIB = Path("/content/Rlib")
R_CACHE = DRIVE_ROOT / "Rlib_colab_cache.tar.gz"

WORKERS = 2

INPUT_DIR.mkdir(parents=True, exist_ok=True)
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

print("BASE_ZIP :", BASE_ZIP)
print("ADDON_ZIP:", ADDON_ZIP)
print("PERSIST  :", PERSIST_ROOT)
print("WORKERS  :", WORKERS)

print("\nCHECK:")
print("BASE ZIP exists :", BASE_ZIP.exists())
print("ADDON ZIP exists:", ADDON_ZIP.exists())

assert BASE_ZIP.exists(), f"File tidak ditemukan: {BASE_ZIP}"
assert ADDON_ZIP.exists(), f"File tidak ditemukan: {ADDON_ZIP}"

print("\n✅ Kedua ZIP ditemukan. Siap lanjut.")

BASE_ZIP : /content/drive/MyDrive/DSDPM_COLAB/input/DSDPM_Reviewer_Revision_v3_4.zip
ADDON_ZIP: /content/drive/MyDrive/DSDPM_COLAB/input/DSDPM_v3_4_COLAB_MULTICORE.zip
PERSIST  : /content/drive/MyDrive/DSDPM_COLAB/persistent
WORKERS  : 2

CHECK:
BASE ZIP exists : True
ADDON ZIP exists: True

✅ Kedua ZIP ditemukan. Siap lanjut.


## 2. Install R dan library sistem

Cell ini perlu dijalankan lagi setiap kali Colab memberi VM baru.


In [ ]:
subprocess.run(["apt-get","update","-qq"], check=True)
subprocess.run([
    "apt-get","install","-y","-qq",
    "r-base","r-base-dev","build-essential","gfortran",
    "libcurl4-openssl-dev","libssl-dev","libxml2-dev",
    "libgdal-dev","libgeos-dev","libproj-dev","libudunits2-dev",
    "libfontconfig1-dev","libharfbuzz-dev","libfribidi-dev",
    "libfreetype6-dev","libpng-dev","libtiff5-dev","libjpeg-dev",
    "libblas-dev","liblapack-dev"
], check=True)
subprocess.run(["Rscript","--version"], check=False)


## 3. Ekstrak paket ke SSD lokal Colab

Komputasi dilakukan di `/content` agar akses source code cepat. Checkpoint hasil tetap disimpan ke Google Drive.


In [ ]:
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)

base_extract = WORK_ROOT / "base"
addon_extract = WORK_ROOT / "addon"
base_extract.mkdir()
addon_extract.mkdir()

with zipfile.ZipFile(BASE_ZIP) as z:
    z.extractall(base_extract)
with zipfile.ZipFile(ADDON_ZIP) as z:
    z.extractall(addon_extract)

candidates = list(base_extract.rglob("MASTER_SOURCE.R"))
assert candidates, "MASTER_SOURCE.R tidak ditemukan dalam ZIP paket asli."
PKG_DIR = candidates[0].parent

# Cari folder addon yang berisi runner.
runner_candidates = list(addon_extract.rglob("RUN_FINAL_BATCH_MULTICORE.R"))
assert runner_candidates, "Runner multicore tidak ditemukan dalam addon ZIP."
ADDON_DIR = runner_candidates[0].parent

for p in ADDON_DIR.iterdir():
    if p.is_file() and p.suffix in {".R", ".md"}:
        shutil.copy2(p, PKG_DIR / p.name)

print("Package directory:", PKG_DIR)
print("Addon installed.")


## 4. Restore/install R packages

Pada session pertama, package R di-install lalu dicache ke Google Drive. Session berikutnya mengembalikan cache yang sama agar versi package tetap konsisten.


In [ ]:
if RLIB.exists():
    shutil.rmtree(RLIB)
RLIB.mkdir(parents=True)

if R_CACHE.exists():
    print("Restoring cached R library...")
    with tarfile.open(R_CACHE, "r:gz") as tf:
        tf.extractall("/content")
else:
    print("No R cache yet; packages will be installed.")

env = os.environ.copy()
env["R_LIBS_USER"] = str(RLIB)
env["DSDPM_PERSIST_ROOT"] = str(PERSIST_ROOT)
env["DSDPM_WORKERS"] = str(WORKERS)

install_script = PKG_DIR / "INSTALL_PACKAGES.R"
if not install_script.exists():
    # Some copies were named INSTALL_PACKAGES(2).R.
    alts = list(PKG_DIR.glob("INSTALL_PACKAGES*.R"))
    assert alts, "INSTALL_PACKAGES.R tidak ditemukan."
    install_script = alts[0]

subprocess.run(
    ["Rscript", str(install_script)],
    cwd=PKG_DIR,
    env=env,
    check=True
)

if not R_CACHE.exists():
    print("Saving R library cache to Drive...")
    with tarfile.open(R_CACHE, "w:gz") as tf:
        tf.add(RLIB, arcname="Rlib")
    print("Cache saved:", R_CACHE)


## 5. Lihat CPU, RAM, dan package versions

Kalau hanya mendapat 2 core, gunakan `WORKERS=2`. Jika mendapat ≥4 core dan RAM cukup, Anda dapat mengubah `WORKERS=4`, lalu jalankan kembali cell ini dan cell batch.


In [ ]:
env["DSDPM_WORKERS"] = str(WORKERS)
subprocess.run(
    ["Rscript", "COLAB_SYSTEM_INFO.R"],
    cwd=PKG_DIR,
    env=env,
    check=True
)


## 6. Validasi environment Colab

**Wajib pada environment pertama.** Script menjalankan urutan validasi asli: model tests → runtime self-test → joint-target test → pilot. Setelah PASS, fingerprint dan certificate disimpan ke Google Drive.

Pada session berikutnya, cell ini hanya memeriksa apakah environment masih identik.


In [ ]:
subprocess.run(
    ["Rscript", "COLAB_VALIDATE_OR_RESTORE.R"],
    cwd=PKG_DIR,
    env=env,
    check=True
)


# Menjalankan Batch

Jalankan **satu batch dahulu**, periksa hasilnya, lalu lanjut ke batch berikutnya.

Jika Colab disconnect, buka notebook lagi, jalankan cell setup 1–6, lalu jalankan **batch yang sama**. Chunk yang sudah tersimpan di Drive akan dilewati.


In [ ]:
# BATCH 1: global replications 1-100
env["DSDPM_BATCH"] = "1"
subprocess.run(
    ["Rscript", "RUN_FINAL_BATCH_MULTICORE.R"],
    cwd=PKG_DIR,
    env=env,
    check=True
)


In [ ]:
# Cek progres / hasil Batch 1
env["DSDPM_BATCH"] = "1"
subprocess.run(
    ["Rscript", "CHECK_COLAB_BATCH_STATUS.R"],
    cwd=PKG_DIR,
    env=env,
    check=True
)


In [ ]:
# BATCH 2: global replications 101-200
env["DSDPM_BATCH"] = "2"
subprocess.run(
    ["Rscript", "RUN_FINAL_BATCH_MULTICORE.R"],
    cwd=PKG_DIR,
    env=env,
    check=True
)


In [ ]:
# BATCH 3: global replications 201-300
env["DSDPM_BATCH"] = "3"
subprocess.run(
    ["Rscript", "RUN_FINAL_BATCH_MULTICORE.R"],
    cwd=PKG_DIR,
    env=env,
    check=True
)


In [ ]:
# BATCH 4: global replications 301-400
env["DSDPM_BATCH"] = "4"
subprocess.run(
    ["Rscript", "RUN_FINAL_BATCH_MULTICORE.R"],
    cwd=PKG_DIR,
    env=env,
    check=True
)


In [ ]:
# BATCH 5: global replications 401-500
env["DSDPM_BATCH"] = "5"
subprocess.run(
    ["Rscript", "RUN_FINAL_BATCH_MULTICORE.R"],
    cwd=PKG_DIR,
    env=env,
    check=True
)


## 7. Merge final R=500

Jalankan hanya setelah kelima batch complete.


In [ ]:
subprocess.run(
    ["Rscript", "RUN_FINAL_MERGE_R500_COLAB.R"],
    cwd=PKG_DIR,
    env=env,
    check=True
)

print("Final output:")
print(PERSIST_ROOT / "final_R500")
